In [26]:
import deepl
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

auth_key = os.getenv("DEEPL_KEY")

if not auth_key:
    raise ValueError("Please set DEEPL_KEY in .env file")

print("✓ DeepL client ready")

✓ DeepL client ready


In [45]:
import json
import time
from pathlib import Path

# File paths
INPUT_FILE = Path("../data/final_quiz_data.json")
OUTPUT_FILE = Path("../data/final_quiz_data.json")
PROGRESS_FILE = Path("../data/translation_progress.json")

print("✓ Setup complete")

✓ Setup complete


## Translation Functions

In [28]:
def translate_to_french(arabic_text):
    """Translate Arabic text to French using DeepL API"""
    try:
        translator = deepl.Translator(auth_key)
        result = translator.translate_text(
            arabic_text,
            source_lang="AR",
            target_lang="FR"
        )
        return result.text, None
    
    except Exception as e:
        return None, str(e)


def load_progress():
    """Load translation progress"""
    if PROGRESS_FILE.exists():
        with open(PROGRESS_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}


def save_progress(progress):
    """Save translation progress"""
    with open(PROGRESS_FILE, 'w', encoding='utf-8') as f:
        json.dump(progress, f, ensure_ascii=False, indent=2)


def save_quiz_data(data):
    """Save updated quiz data immediately"""
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

print("✓ Functions ready")

✓ Functions ready


## Load Quiz Data

In [29]:
# Load quiz data
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    quiz_data = json.load(f)

# Load progress
progress = load_progress()

print(f"Loaded {len(quiz_data)} tests")
print(f"Translation progress: {len(progress)} items completed")

Loaded 20 tests
Translation progress: 0 items completed


## Count Total Items to Translate

In [47]:
# Count items
total_signs = 0
total_priorities = 0
total_questions = 0

for test_name, test_data in quiz_data.items():
    sections = test_data.get('sections', {})
    total_signs += len(sections.get('road_signs', []))
    total_priorities += len(sections.get('priorities', []))
    total_questions += len(sections.get('general_questions', []))

total_items = total_signs + total_priorities + total_questions

print(f"Total items to translate:")
print(f"  Road Signs: {total_signs}")
print(f"  Priorities: {total_priorities}")
print(f"  General Questions: {total_questions}")
print(f"  TOTAL: {total_items}")

Total items to translate:
  Road Signs: 320
  Priorities: 160
  General Questions: 119
  TOTAL: 599


## Test Translation on First Test Only

In [31]:
print("Testing translation on test-01 only...\n")

test_name = 'test-01'
test_data = quiz_data[test_name]
sections = test_data.get('sections', {})

# Test Road Signs
print("Road Signs:")
for sign in sections.get('road_signs', [])[:3]:  # Test first 3
    arabic_answer = sign.get('answer_ar', '')
    current_french = sign.get('answer_fr', '')
    
    print(f"\nSign {sign['sign_number']}:")
    print(f"  Arabic: {arabic_answer[:100]}...")
    print(f"  Current French: {current_french}")
    
    if arabic_answer and (not current_french or current_french == '###'):
        print(f"  Translating...", end=" ", flush=True)
        french_answer, error = translate_to_french(arabic_answer)
        
        if error:
            print(f"✗ Error: {error}")
        else:
            print(f"✓")
            print(f"  New French: {french_answer[:100]}...")
            sign['answer_fr'] = french_answer
        
        time.sleep(5)

# Test Priorities
print("\n\nPriorities:")
for priority in sections.get('priorities', [])[:2]:  # Test first 2
    arabic_answer = priority.get('answer_ar', '')
    current_french = priority.get('answer_fr', '')
    
    print(f"\nPriority {priority['priority_number']}:")
    print(f"  Arabic: {arabic_answer[:100]}...")
    print(f"  Current French: {current_french}")
    
    if arabic_answer and (not current_french or current_french == '###'):
        print(f"  Translating...", end=" ", flush=True)
        french_answer, error = translate_to_french(arabic_answer)
        
        if error:
            print(f"✗ Error: {error}")
        else:
            print(f"✓")
            print(f"  New French: {french_answer[:100]}...")
            priority['answer_fr'] = french_answer
        
        time.sleep(5)

# Test General Questions
print("\n\nGeneral Questions:")
for question in sections.get('general_questions', [])[:2]:  # Test first 2
    arabic_answer = question.get('answer_ar', '')
    current_french = question.get('answer_fr', '')
    
    print(f"\nQuestion {question['question_number']}:")
    print(f"  Arabic: {arabic_answer[:100]}...")
    print(f"  Current French: {current_french}")
    
    if arabic_answer and (not current_french or current_french == '###'):
        print(f"  Translating...", end=" ", flush=True)
        french_answer, error = translate_to_french(arabic_answer)
        
        if error:
            print(f"✗ Error: {error}")
        else:
            print(f"✓")
            print(f"  New French: {french_answer[:100]}...")
            question['answer_fr'] = french_answer
        
        time.sleep(5)

print("\n\n" + "="*60)
print("TEST COMPLETE - Review the translations above")
print("If they look good, proceed to translate all tests")
print("="*60)

Testing translation on test-01 only...

Road Signs:

Sign 1:
  Arabic: 1- حذار، خطر غير معين....
  Current French: ###
  Translating... ✓
  New French: 1- Attention, risque non spécifié....
✓
  New French: 1- Attention, risque non spécifié....

Sign 2:
  Arabic: 2- الدوران إلى اليمين ممنوع....
  Current French: ###
  Translating... 
Sign 2:
  Arabic: 2- الدوران إلى اليمين ممنوع....
  Current French: ###
  Translating... ✓
  New French: 2- Il est interdit de tourner à droite....
✓
  New French: 2- Il est interdit de tourner à droite....

Sign 3:
  Arabic: 3- التوقف ممنوع من 16 لـ 31 من كل شهر....
  Current French: ###
  Translating... 
Sign 3:
  Arabic: 3- التوقف ممنوع من 16 لـ 31 من كل شهر....
  Current French: ###
  Translating... ✓
  New French: 3- L'arrêt est interdit du 16 au 31 de chaque mois....
✓
  New French: 3- L'arrêt est interdit du 16 au 31 de chaque mois....


Priorities:

Priority 1:
  Arabic: 1- محور دوراني مع إشارة ترك المرور - تمر السيارة الحمراء ثم الصفراء ....
  Curr

## Translate Road Signs Answers

In [48]:
print("Translating Road Signs...\n")
signs_translated = 0
signs_skipped = 0
signs_failed = 0

for test_name in sorted(quiz_data.keys()):
    test_data = quiz_data[test_name]
    road_signs = test_data.get('sections', {}).get('road_signs', [])
    
    for i, sign in enumerate(road_signs):
        # Create unique ID for tracking
        item_id = f"{test_name}_sign_{sign['sign_number']}"
        
        # Get current answer
        arabic_answer = sign.get('answer_ar', '')
        current_french = sign.get('answer_fr', '')
        
        # Skip if already translated (not a placeholder)
        if current_french and current_french != '###':
            signs_skipped += 1
            continue
        
        # Skip if in progress cache
        if item_id in progress:
            sign['answer_fr'] = progress[item_id]
            signs_skipped += 1
            continue
        
        # Skip if no Arabic text
        if not arabic_answer:
            print(f"⚠ {test_name} Sign {sign['sign_number']}: No Arabic text, skipping")
            signs_skipped += 1
            continue
        
        print(f"→ {test_name} Sign {sign['sign_number']}: Translating...", end=" ", flush=True)
        
        french_answer, error = translate_to_french(arabic_answer)
        
        if error:
            print(f"✗ Error: {error}")
            signs_failed += 1
            time.sleep(5)
            continue
        
        # Update data structure
        sign['answer_fr'] = french_answer
        
        # Save immediately
        progress[item_id] = french_answer
        save_progress(progress)
        save_quiz_data(quiz_data)
        
        signs_translated += 1
        print("✓")
        time.sleep(5)  # Rate limit

print(f"\n{'='*60}")
print(f"Road Signs: {signs_translated} translated, {signs_skipped} skipped, {signs_failed} failed")
print(f"{'='*60}")

Translating Road Signs...

→ test-01 Sign 1: Translating... ✓
✓
→ test-01 Sign 2: Translating... → test-01 Sign 2: Translating... ✓
✓
→ test-01 Sign 3: Translating... → test-01 Sign 3: Translating... ✓
✓

Road Signs: 3 translated, 317 skipped, 0 failed

Road Signs: 3 translated, 317 skipped, 0 failed


## Translate Priorities Answers

In [49]:
print("Translating Priorities...\n")
priorities_translated = 0
priorities_skipped = 0
priorities_failed = 0

for test_name in sorted(quiz_data.keys()):
    test_data = quiz_data[test_name]
    priorities = test_data.get('sections', {}).get('priorities', [])
    
    for i, priority in enumerate(priorities):
        # Create unique ID
        item_id = f"{test_name}_priority_{priority['priority_number']}"
        
        # Get current answer
        arabic_answer = priority.get('answer_ar', '')
        current_french = priority.get('answer_fr', '')
        
        # Skip if already translated (not a placeholder)
        if current_french and current_french != '###':
            priorities_skipped += 1
            continue
        
        # Skip if in progress cache
        if item_id in progress:
            priority['answer_fr'] = progress[item_id]
            priorities_skipped += 1
            continue
        
        # Skip if no Arabic text
        if not arabic_answer:
            print(f"⚠ {test_name} Priority {priority['priority_number']}: No Arabic text, skipping")
            priorities_skipped += 1
            continue
        
        print(f"→ {test_name} Priority {priority['priority_number']}: Translating...", end=" ", flush=True)
        
        french_answer, error = translate_to_french(arabic_answer)
        
        if error:
            print(f"✗ Error: {error}")
            priorities_failed += 1
            time.sleep(5)
            continue
        
        # Update data structure
        priority['answer_fr'] = french_answer
        
        # Save immediately
        progress[item_id] = french_answer
        save_progress(progress)
        save_quiz_data(quiz_data)
        
        priorities_translated += 1
        print("✓")
        time.sleep(5)  # Rate limit

print(f"\n{'='*60}")
print(f"Priorities: {priorities_translated} translated, {priorities_skipped} skipped, {priorities_failed} failed")
print(f"{'='*60}")

Translating Priorities...

→ test-01 Priority 1: Translating... ✓
✓
→ test-01 Priority 2: Translating... → test-01 Priority 2: Translating... ✓
✓

Priorities: 2 translated, 158 skipped, 0 failed

Priorities: 2 translated, 158 skipped, 0 failed


## Translate General Questions Answers

In [50]:
print("Translating General Questions...\n")
questions_translated = 0
questions_skipped = 0
questions_failed = 0

for test_name in sorted(quiz_data.keys()):
    test_data = quiz_data[test_name]
    questions = test_data.get('sections', {}).get('general_questions', [])
    
    for i, question in enumerate(questions):
        # Create unique ID
        item_id = f"{test_name}_question_{question['question_number']}"
        
        # Get current answer
        arabic_answer = question.get('answer_ar', '')
        current_french = question.get('answer_fr', '')
        
        # Skip if already translated (not a placeholder)
        if current_french and current_french != '###':
            questions_skipped += 1
            continue
        
        # Skip if in progress cache
        if item_id in progress:
            question['answer_fr'] = progress[item_id]
            questions_skipped += 1
            continue
        
        # Skip if no Arabic text
        if not arabic_answer:
            print(f"⚠ {test_name} Question {question['question_number']}: No Arabic text, skipping")
            questions_skipped += 1
            continue
        
        print(f"→ {test_name} Question {question['question_number']}: Translating...", end=" ", flush=True)
        
        french_answer, error = translate_to_french(arabic_answer)
        
        if error:
            print(f"✗ Error: {error}")
            questions_failed += 1
            time.sleep(5)
            continue
        
        # Update data structure
        question['answer_fr'] = french_answer
        
        # Save immediately
        progress[item_id] = french_answer
        save_progress(progress)
        save_quiz_data(quiz_data)
        
        questions_translated += 1
        print("✓")
        time.sleep(5)  # Rate limit

print(f"\n{'='*60}")
print(f"General Questions: {questions_translated} translated, {questions_skipped} skipped, {questions_failed} failed")
print(f"{'='*60}")

Translating General Questions...

→ test-01 Question 1: Translating... ✓
✓
→ test-01 Question 2: Translating... → test-01 Question 2: Translating... ✓
✓

General Questions: 2 translated, 117 skipped, 0 failed

General Questions: 2 translated, 117 skipped, 0 failed


## Sync Progress to Output File

In [51]:
print("Syncing all translations from progress file to output file...\n")

# Load the latest progress
progress = load_progress()
print(f"Found {len(progress)} translations in progress file")

# Load or create output data
if OUTPUT_FILE.exists():
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        quiz_data = json.load(f)
    print(f"Loaded existing output file with {len(quiz_data)} tests")
else:
    # Start from input file
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        quiz_data = json.load(f)
    print(f"Created new output from input file with {len(quiz_data)} tests")

# Apply all translations from progress
synced_count = 0
missing_count = 0

for item_id, french_translation in progress.items():
    # Parse item_id: "test-01_sign_4" or "test-01_priority_3" or "test-01_question_5"
    parts = item_id.split('_')
    test_name = parts[0]  # e.g., "test-01"
    item_type = parts[1]  # e.g., "sign", "priority", "question"
    item_number = int(parts[2])  # e.g., 4
    
    # Navigate to the correct section
    if test_name not in quiz_data:
        print(f"⚠ Warning: Test {test_name} not found in quiz data")
        missing_count += 1
        continue
    
    sections = quiz_data[test_name].get('sections', {})
    
    # Determine the section name
    if item_type == 'sign':
        section_name = 'road_signs'
        id_field = 'sign_number'
    elif item_type == 'priority':
        section_name = 'priorities'
        id_field = 'priority_number'
    elif item_type == 'question':
        section_name = 'general_questions'
        id_field = 'question_number'
    else:
        print(f"⚠ Warning: Unknown item type '{item_type}' in {item_id}")
        missing_count += 1
        continue
    
    # Find and update the item
    items = sections.get(section_name, [])
    found = False
    
    for item in items:
        if item.get(id_field) == item_number:
            item['answer_fr'] = french_translation
            synced_count += 1
            found = True
            break
    
    if not found:
        print(f"⚠ Warning: Item {item_id} not found in quiz data")
        missing_count += 1

# Save the synced data
save_quiz_data(quiz_data)

print(f"\n{'='*60}")
print(f"SYNC COMPLETE")
print(f"{'='*60}")
print(f"Synced: {synced_count} translations")
print(f"Missing: {missing_count} items")
print(f"Output saved to: {OUTPUT_FILE}")
print(f"{'='*60}")

Syncing all translations from progress file to output file...

Found 594 translations in progress file
Loaded existing output file with 20 tests

SYNC COMPLETE
Synced: 594 translations
Missing: 0 items
Output saved to: ..\data\final_quiz_data.json


## Final Summary & Verification

In [52]:
# Reload data to verify
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    final_data = json.load(f)

# Count completeness
total_complete = 0
total_incomplete = 0

for test_name, test_data in final_data.items():
    sections = test_data.get('sections', {})
    
    for sign in sections.get('road_signs', []):
        answer_fr = sign.get('answer_fr', '')
        if answer_fr and answer_fr != '###':
            total_complete += 1
        else:
            total_incomplete += 1
    
    for priority in sections.get('priorities', []):
        answer_fr = priority.get('answer_fr', '')
        if answer_fr and answer_fr != '###':
            total_complete += 1
        else:
            total_incomplete += 1
    
    for question in sections.get('general_questions', []):
        answer_fr = question.get('answer_fr', '')
        if answer_fr and answer_fr != '###':
            total_complete += 1
        else:
            total_incomplete += 1

print(f"\n{'='*60}")
print(f"FINAL SUMMARY")
print(f"{'='*60}")
print(f"Total items: {total_items}")
print(f"Completed: {total_complete} ({total_complete/total_items*100:.1f}%)")
print(f"Incomplete: {total_incomplete}")
print(f"\nTotal translated this session:")
print(f"  Road Signs: {signs_translated}")
print(f"  Priorities: {priorities_translated}")
print(f"  Questions: {questions_translated}")
print(f"  TOTAL: {signs_translated + priorities_translated + questions_translated}")
print(f"\nTotal failed:")
print(f"  Road Signs: {signs_failed}")
print(f"  Priorities: {priorities_failed}")
print(f"  Questions: {questions_failed}")
print(f"  TOTAL: {signs_failed + priorities_failed + questions_failed}")
print(f"{'='*60}")

if total_incomplete > 0:
    print(f"\n⚠️  Run this notebook again to translate remaining {total_incomplete} items")
else:
    print(f"\n✓ ALL TRANSLATIONS COMPLETE!")


FINAL SUMMARY
Total items: 599
Completed: 599 (100.0%)
Incomplete: 0

Total translated this session:
  Road Signs: 3
  Priorities: 2
  Questions: 2
  TOTAL: 7

Total failed:
  Road Signs: 0
  Priorities: 0
  Questions: 0
  TOTAL: 0

✓ ALL TRANSLATIONS COMPLETE!


## View Sample Translations

In [54]:
# Show sample from first test
first_test = final_data['test-01']
sections = first_test['sections']

print("Sample Road Sign:")
if sections['road_signs']:
    sign = sections['road_signs'][0]
    print(f"Sign #{sign['sign_number']}")
    print(f"Arabic: {sign.get('answer_ar', 'N/A')[:80]}...")
    print(f"French: {sign.get('answer_fr', 'N/A')[:80]}...")

print("\nSample Priority:")
if sections['priorities']:
    priority = sections['priorities'][0]
    print(f"Priority #{priority['priority_number']}")
    print(f"Arabic: {priority.get('answer_ar', 'N/A')[:80]}...")
    print(f"French: {priority.get('answer_fr', 'N/A')[:80]}...")

print("\nSample General Question:")
if sections['general_questions']:
    q = sections['general_questions'][0]
    print(f"Question #{q['question_number']}")
    print(f"Question FR: {q.get('question_fr', 'N/A')[:80]}...")
    print(f"Answer AR: {q.get('answer_ar', 'N/A')[:80]}...")
    print(f"Answer FR: {q.get('answer_fr', 'N/A')[:80]}...")

Sample Road Sign:
Sign #1
Arabic: 1- حذار، خطر غير معين....
French: 1- Attention, risque non spécifié....

Sample Priority:
Priority #1
Arabic: 1- محور دوراني مع إشارة ترك المرور - تمر السيارة الحمراء ثم الصفراء ....
French: 1- Axe de rotation avec un feu de circulation - la voiture rouge passe, puis la ...

Sample General Question:
Question #1
Question FR: De quels facteurs dépend la distance de sécurité ?...
Answer AR: 1- تتعلق مسافة الأمان بـ:
- السرعة، كلما زادت سرعة السيارة زادت مسافة الأمان.
- ...
Answer FR: 1- La distance de sécurité est liée à :
- La vitesse, plus la vitesse du véhicul...
